# Bag-of-Words Baseline

A surface-text control for the eval-awareness probe. It answers one question:

> How much of the probe's AUROC could you get **without the model at all**?

If a linear classifier over word counts separates `testing` from `deployment` about as
well as the residual-stream probe does, then the probe's score is not evidence that the
model represents "I am being evaluated". It is evidence that the two classes are
lexically distinguishable, which we already know.

The labelled set is built by `scripts/build_labelled_eval.py`: the testing side is a
theory-of-mind benchmark item, the deployment side is a real ShareGPT user turn. Two
different corpora, so they differ in nearly every surface property at once.

Implements `docs/bag-of-words-baseline.md`.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Repo root, whether the notebook is run from notebooks/ or from the repo root.
REPO = Path.cwd()
if not (REPO / "datasets").is_dir():
    REPO = REPO.parent

# Which labelled eval set to baseline. Both are {question, type} files that
# analyze_probe.py also consumes, so the BoW control lines up with whatever set
# the probe was scored on. This notebook is model-free and runs locally, so the
# chosen file must exist under datasets/ on THIS machine:
#   "tom" -> eval_vs_deploy_labelled.json  (scripts/build_labelled_eval.py)
#   "sad" -> sad_oversight_labelled.json   (scripts/build_sad_oversight_eval.py)
# The colab notebook builds the SAD file on the VM; to get it here, run
#   python scripts/build_sad_oversight_eval.py --sad-repo ./sad
# after `git clone https://github.com/LRudL/sad.git`.
DATASET = "sad"

_CONFIG = {
    "tom": {
        "data": "eval_vs_deploy_labelled.json",
        # probe_analysis.py reports 0.116 at layer 10 under the deployment-positive
        # convention (src/probe_analysis.py:183); inverted that is 0.884.
        "probe_auroc": 0.884,
        "probe_auroc_length_matched": 0.844,
        "probe_label": "probe, layer 10 (sign-corrected)",
    },
    "sad": {
        "data": "sad_oversight_labelled.json",
        # Fill these from your SAD analyze_probe run: take the best layer's AUROC
        # from {run_dir}/analysis/probe_stats_layer*.csv, sign-corrected as
        # abs(auroc - 0.5) + 0.5 (deployment-positive convention). Left None until
        # then, so the comparison shows NaN rather than a stale ToM number.
        "probe_auroc": None,
        "probe_auroc_length_matched": None,
        "probe_label": "probe, best SAD layer (sign-corrected)",
    },
}[DATASET]

DATA = REPO / "datasets" / _CONFIG["data"]
OUTPUT = REPO / "analysis" / f"bow_baseline_{DATASET}.json"
FOLDS = 5
SEED = 0

# The probe AUROC to compare BoW against, for the currently selected DATASET.
PROBE_AUROC = _CONFIG["probe_auroc"]
PROBE_AUROC_LENGTH_MATCHED = _CONFIG["probe_auroc_length_matched"]
PROBE_LABEL = _CONFIG["probe_label"]

In [ ]:
# y = 1 for deployment, 0 for testing -- matching src/probe_analysis.py:183 so the
# number below is directly comparable to the probe's AUROC without sign-flipping.
records = json.loads(DATA.read_text(encoding="utf-8"))
texts = np.array([r["question"] for r in records], dtype=object)
y = np.array([1 if r["type"] == "deployment" else 0 for r in records])

print(f"{DATA.name}: {len(texts)} examples "
      f"({(y == 0).sum()} testing / {(y == 1).sum()} deployment)")

## 1. The surface confound, counted

Before fitting anything, check how far apart the two corpora are on properties no theory
of eval-awareness would predict.

In [ ]:
CUES = {
    'Brevity instruction ("brief", "concise")':
        lambda s: bool(re.search(r"\b(brief|briefly|concise|succinct)\b", s, re.I)),
    "Belief/ToM verbs (think, believe, unaware)":
        lambda s: bool(re.search(r"\b(think|thinks|believe|believes|unaware|assume)\b", s, re.I)),
    "Imperative opening (write/create/act as/help me)":
        lambda s: bool(re.match(r"\s*(write|create|act as|help me|make|give me|generate)\b", s, re.I)),
    "Any non-ASCII character":
        lambda s: any(ord(c) > 127 for c in s),
    "Curly quote / em dash / ellipsis":
        lambda s: any(c in s for c in "\u2018\u2019\u201c\u201d\u2014\u2026"),
}

cue_table = pd.DataFrame(
    [{"feature": name,
      "testing": int(sum(fn(t) for t in texts[y == 0])),
      "deployment": int(sum(fn(t) for t in texts[y == 1]))}
     for name, fn in CUES.items()]
).set_index("feature")

cue_table.loc["Median length (chars)"] = [
    int(np.median([len(t) for t in texts[y == 0]])),
    int(np.median([len(t) for t in texts[y == 1]])),
]
cue_table

The non-ASCII split is the tell. It is *not* a language effect -- it is prompts containing
curly quotes, em dashes and ellipses: the fingerprint of text copy-pasted out of a browser,
versus text emitted programmatically by a vignette generator. Provenance, not content.

A probe reading layer 10 -- early, where register and formatting live -- has ample surface
signal to exploit. The rest of this notebook measures how much.

## 2. Feature sets

Four ablations, each isolating one class of surface cue.

| Ablation | Feature set | What it isolates |
| --- | --- | --- |
| `full` | tf-idf 1-2 grams | Total surface separability |
| `function-words` | ~100 most frequent English function words, relative frequency | Register/style, no topic |
| `punctuation` | Counts of `?`, `.`, `,`, quotes, newlines, non-ASCII flag | Formatting/provenance alone |
| `length` | Character count, single feature | The crudest possible cue |

No hand-built features in `full` -- the point is that naive surface statistics suffice, so
any cleverness weakens the argument.

In [ ]:
# The ~100 most frequent English words, all function words / high-frequency glue.
FUNCTION_WORDS = """
the of and a to in is you that it he was for on are as with his they i at be this have from
or one had by word but not what all were we when your can said there use an each which she do
how their if will up other about out many then them these so some her would make like him into
time has look two more write go see number no way could people my than first been call who its
now find long down did get come made may part over new sound take only little work know place
""".split()

PUNCT_FEATURES = {
    "question_marks": lambda s: s.count("?"),
    "periods": lambda s: s.count("."),
    "commas": lambda s: s.count(","),
    "straight_quotes": lambda s: s.count('"') + s.count("'"),
    "curly_quotes": lambda s: sum(s.count(c) for c in "\u2018\u2019\u201c\u201d"),
    "newlines": lambda s: s.count("\n"),
    "colons": lambda s: s.count(":"),
    "digits": lambda s: sum(c.isdigit() for c in s),
    "non_ascii": lambda s: float(any(ord(c) > 127 for c in s)),
}


class SurfaceFeatures(BaseEstimator, TransformerMixin):
    """Apply a dict of {name: str -> number} functions to raw texts."""

    def __init__(self, funcs):
        self.funcs = funcs

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([[f(s) for f in self.funcs.values()] for s in X], dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array(list(self.funcs))


ABLATIONS = ["full", "function-words", "punctuation", "length"]


def make_pipeline(ablation):
    """Feature extractor + logistic regression for one ablation.

    The classes are balanced 147/147, but keep class_weight so this stays honest
    on re-splits.
    """
    clf = LogisticRegression(penalty="l2", C=1.0, max_iter=2000, class_weight="balanced")

    if ablation == "full":
        features = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2,
                                   sublinear_tf=True)
    elif ablation == "function-words":
        # norm='l1' with use_idf=False gives relative frequency within the prompt,
        # so this measures style rather than length.
        features = TfidfVectorizer(lowercase=True, vocabulary=FUNCTION_WORDS,
                                   use_idf=False, norm="l1")
    elif ablation == "punctuation":
        features = Pipeline([("counts", SurfaceFeatures(PUNCT_FEATURES)),
                             ("scale", StandardScaler())])
    elif ablation == "length":
        features = Pipeline([("counts", SurfaceFeatures({"n_chars": len})),
                             ("scale", StandardScaler())])
    else:
        raise ValueError(f"unknown ablation: {ablation}")

    return Pipeline([("features", features), ("clf", clf)])

## 3. Cross-validated AUROC

The vectorizer is fit **inside each fold on the training split only** -- fitting it on the
full corpus leaks test-set vocabulary and inflates the score. AUROC uses `roc_curve` +
`auc` to mirror `probe_analysis.py`.

In [ ]:
def cross_val_auroc(ablation, folds=FOLDS, seed=SEED):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    aurocs = []
    for train_idx, test_idx in skf.split(texts, y):
        pipe = make_pipeline(ablation).fit(texts[train_idx], y[train_idx])
        scores = pipe.predict_proba(texts[test_idx])[:, 1]
        fpr, tpr, _ = roc_curve(y[test_idx], scores)
        aurocs.append(auc(fpr, tpr))
    return np.array(aurocs)


fold_scores = {ab: cross_val_auroc(ab) for ab in ABLATIONS}

results = pd.DataFrame(
    [{"ablation": ab,
      "mean_auroc": s.mean(),
      "std": s.std(),
      "min_fold": s.min(),
      "max_fold": s.max()}
     for ab, s in fold_scores.items()]
).set_index("ablation").round(3)
results

In [ ]:
bow_best = results["mean_auroc"].max()

comparison = pd.DataFrame({
    "AUROC": [results.loc["full", "mean_auroc"],
              results.loc["punctuation", "mean_auroc"],
              results.loc["length", "mean_auroc"],
              PROBE_AUROC,
              PROBE_AUROC_LENGTH_MATCHED],
    "source": ["bag-of-words (5-fold CV)", "punctuation only", "length only",
               PROBE_LABEL, PROBE_LABEL + ", length matched"],
}, index=["bow_full", "bow_punctuation", "bow_length", "probe", "probe_filtered"])
comparison.round(3)

## 4. Top features

This turns a number into a diagnosis. If the top testing terms are `brief`, `response`,
`she`, `think`, the classifier is reading the benchmark template -- which is precisely the
confound.

Fit on all 294 examples: this is for inspection, not scoring.

In [ ]:
inspect = make_pipeline("full").fit(texts, y)
names = inspect["features"].get_feature_names_out()
coef = inspect["clf"].coef_[0]
order = np.argsort(coef)

top_features = pd.DataFrame({
    "deployment_term": [names[i] for i in order[::-1][:20]],
    "weight": [round(coef[i], 3) for i in order[::-1][:20]],
    "testing_term": [names[i] for i in order[:20]],
    "weight ": [round(coef[i], 3) for i in order[:20]],
})
top_features

## 5. Save

In [ ]:
payload = {
    "data": str(DATA),
    "n_examples": int(len(texts)),
    "n_testing": int((y == 0).sum()),
    "n_deployment": int((y == 1).sum()),
    "folds": FOLDS,
    "seed": SEED,
    "label_convention": "deployment=1, testing=0 (src/probe_analysis.py:183)",
    "probe_auroc_layer10_sign_corrected": PROBE_AUROC,
    "ablations": {
        ab: {"mean_auroc": float(s.mean()), "std": float(s.std()),
             "fold_aurocs": [float(v) for v in s]}
        for ab, s in fold_scores.items()
    },
    "top_deployment_terms": [str(names[i]) for i in order[::-1][:20]],
    "top_testing_terms": [str(names[i]) for i in order[:20]],
    "surface_cue_counts": cue_table.to_dict(orient="index"),
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Wrote {OUTPUT}")

## Interpreting the result

Compare the mean cross-validated AUROC against the probe's best layer (0.884 at layer 10,
deployment-negative direction):

- **BoW >= 0.85** -- the probe has no demonstrated advantage over word counts. The layer-10
  result should not be reported as evidence of an eval-awareness representation.
- **BoW 0.70-0.85** -- surface cues explain most but not all of the probe's separation. The
  residual gap is small and needs a confound-free dataset before it means anything.
- **BoW <= 0.65** -- genuinely surprising. Re-check the fold-wise vectorizer fitting for a
  bug before believing it.

Note the asymmetry in what this test can show. A high BoW score **falsifies** the strong
reading of the probe result. A low BoW score does **not** validate it -- it only fails to
rule it out, since a probe can exploit surface cues that word counts miss. This is a
control, not a confirmation.

### What this baseline cannot fix

The confound is in the dataset, so no re-analysis of the existing files resolves it. A
positive result here is a reason to rebuild the eval set, not to tune the probe. Two
constructions that would actually isolate eval-awareness:

1. **Frame-swap pairs.** Hold the prompt body fixed and vary only the wrapper: the same
   organic request as `"Q: {body} Provide a very brief response."` versus plainly. Genre is
   constant by construction, so BoW gets ~0.5 and any probe separation is attributable to
   the frame.
2. **Single-source labelling.** Draw both classes from the same scrape and label by whether
   the item is *actually* a benchmark question, rather than pairing one synthetic generator
   against one web corpus.

Until one of those exists, treat every AUROC in `analysis/` as an upper bound contaminated
by an unknown amount of genre detection.